# Exercise 3: HADDOCK3 Protein-Protein Docking

This notebook runs a protein-protein docking example using **HADDOCK3** — the modular BioExcel docking platform.

**System:** E2A (glucose-specific enzyme IIA) + HPr (histidine-containing phosphocarrier protein)  
**Restraints:** NMR chemical shift perturbation data (AIRs)  
**Reference complex:** PDB 1GGR

---
⚠️ **Runtime:** Set to **GPU** or at minimum **High-RAM** in Colab for acceptable speed.  
⚠️ HADDOCK3 requires **Python 3.9+** and will install CNS binaries automatically.

## 1. Install HADDOCK3

In [ ]:
# Install HADDOCK3 and py3Dmol for visualization
!pip install -qqq haddock3 py3Dmol

## 2. Mount Google Drive & Set Up Working Directory

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

exercise_path = '/content/drive/MyDrive/datasmiths_bootcamp/day_2/exercise_haddock3'
os.makedirs(exercise_path, exist_ok=True)
os.chdir(exercise_path)

# Create data directory for input PDB and restraint files
data_dir = os.path.join(exercise_path, 'data')
os.makedirs(data_dir, exist_ok=True)

print(f'Working directory: {exercise_path}')

## 3. Download Input Files

Files are pulled directly from the official HADDOCK3 GitHub examples:
- `e2aP_1F3G.pdb` — E2A protein (chain A), phosphorylated HIS90
- `hpr_ensemble.pdb` — HPr NMR ensemble (10 conformations)
- `e2a-hpr_air.tbl` — Ambiguous Interaction Restraints (AIRs) from NMR CSP data
- `e2a-hpr_1GGR.pdb` — Reference complex structure for CAPRI evaluation

In [ ]:
import urllib.request

BASE_URL = 'https://raw.githubusercontent.com/haddocking/haddock3/main/examples/docking-protein-protein/data/'

files = [
    'e2aP_1F3G.pdb',
    'hpr_ensemble.pdb',
    'e2a-hpr_air.tbl',
    'e2a-hpr_1GGR.pdb',
]

for fname in files:
    dest = os.path.join(data_dir, fname)
    if not os.path.exists(dest):
        urllib.request.urlretrieve(BASE_URL + fname, dest)
        print(f'Downloaded: {fname}')
    else:
        print(f'Already exists: {fname}')

print('\nFiles in data/:')
print(os.listdir(data_dir))

## 4. Inspect Input Structures

Quick look at E2A and HPr before docking.

In [ ]:
import py3Dmol

view = py3Dmol.view(viewergrid=(1, 2), width=1000, height=400)

for i, (fname, label, color) in enumerate([
    ('e2aP_1F3G.pdb', 'E2A (1F3G)', 'yellow'),
    ('hpr_ensemble.pdb', 'HPr ensemble (1HDN)', 'cyan'),
]):
    with open(os.path.join(data_dir, fname)) as f:
        pdb_data = f.read()
    view.addModel(pdb_data, 'pdb', viewer=(0, i))
    view.setStyle({'cartoon': {'color': color}}, viewer=(0, i))
    view.zoomTo(viewer=(0, i))
    print(f'Loaded: {label}')

view.show()

## 5. Write HADDOCK3 Configuration File

This config reproduces the classic HADDOCK2.x rigid-body + flexible-refinement + energy-minimisation workflow:

| Step | Module | Purpose |
|------|--------|---------|
| 0 | `topoaa` | Generate CNS topology |
| 1 | `rigidbody` | 50 rigid-body docking models (reduced for Colab speed) |
| 2 | `caprieval` | Evaluate vs. reference |
| 3 | `seletop` | Select top 10 |
| 4 | `flexref` | Flexible refinement |
| 5 | `emref` | Energy minimisation |
| 6 | `clustfcc` | FCC clustering |
| 7 | `caprieval` | Final evaluation |

In [ ]:
cfg_content = f"""# HADDOCK3 protein-protein docking — E2A/HPr example
# Reduced sampling (50 models) for Colab; increase for production runs.

run_dir = "{exercise_path}/run1-e2a-hpr"
ncores = 2
mode = "local"

molecules = [
    "{data_dir}/e2aP_1F3G.pdb",
    "{data_dir}/hpr_ensemble.pdb"
]

[topoaa]
autohis = false

[topoaa.mol1]
nhisd = 0
nhise = 1
hise_1 = 75

[topoaa.mol2]
nhisd = 1
hisd_1 = 76
nhise = 1
hise_1 = 15

[rigidbody]
tolerance = 5
ambig_fname = "{data_dir}/e2a-hpr_air.tbl"
sampling = 10

[caprieval]
reference_fname = "{data_dir}/e2a-hpr_1GGR.pdb"

[seletop]
select = 1

[flexref]
tolerance = 5
ambig_fname = "{data_dir}/e2a-hpr_air.tbl"

[emref]
tolerance = 5
ambig_fname = "{data_dir}/e2a-hpr_air.tbl"

[clustfcc]

[caprieval]
reference_fname = "{data_dir}/e2a-hpr_1GGR.pdb"
"""

cfg_path = os.path.join(exercise_path, 'docking-e2a-hpr.cfg')
with open(cfg_path, 'w') as f:
    f.write(cfg_content)

print(f'Config written to: {cfg_path}')
print('\n--- Config preview ---')
print(cfg_content)

## 6. Run HADDOCK3

This will take **~2 min** on Colab with 10 rigid-body models. For a full 1000-model production run, increase `sampling = 1000` and `select = 200` in the config above.

In [ ]:
!haddock3 {cfg_path}

## 7. Check Results

HADDOCK3 outputs a structured run directory. The final CAPRI evaluation tells you model quality vs. the known complex.

In [ ]:
import pandas as pd
from pathlib import Path

run_dir = Path(exercise_path) / 'run1-e2a-hpr'

# List all step directories
steps = sorted([d for d in run_dir.iterdir() if d.is_dir()])
print('Run directory contents:')
for s in steps:
    print(f'  {s.name}/')

# Find the final caprieval directory (last one)
capri_dirs = sorted(run_dir.glob('*_caprieval'))
final_capri = capri_dirs[-1] if capri_dirs else None

if final_capri:
    print(f'\nFinal CAPRI results from: {final_capri.name}')
    capri_ss = final_capri / 'capri_ss.tsv'
    if capri_ss.exists():
        df = pd.read_csv(capri_ss, sep='\t')
        print(df[['model', 'score', 'irmsd', 'lrmsd', 'fnat', 'dockq', 'bsa', 'desolv', 'elec', 'vdw', 'total']].head(10).to_string(index=False))

In [ ]:
import gzip
import shutil
from pathlib import Path

run_dir = Path(exercise_path) / 'run1-e2a-hpr'

for gz_file in run_dir.rglob('*.pdb.gz'):
    out_file = gz_file.with_suffix('')  # removes .gz → keeps .pdb
    with gzip.open(gz_file, 'rb') as f_in, open(out_file, 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)
    print(f'Decompressed: {gz_file.name} → {out_file.name}')

## 8. Visualize Top Docking Models

Display the top models from the final refinement step alongside the reference complex (1GGR).
- **Yellow** = E2A (chain A)
- **Cyan** = HPr (chain B)
- **Orange** = Reference complex (1GGR)

In [ ]:
import py3Dmol
from pathlib import Path

run_dir = Path(exercise_path) / 'run1-e2a-hpr'

# Get top models from final emref step
emref_dirs = sorted(run_dir.glob('*_emref'))
if not emref_dirs:
    emref_dirs = sorted(run_dir.glob('*_flexref'))  # fallback

pdb_files = []
if emref_dirs:
    pdb_files = sorted(emref_dirs[-1].glob('*.pdb'))

n_models = len(pdb_files)
if n_models == 0:
    print('No PDB files found — check that the run completed successfully.')
else:
    cols = min(n_models, 2)
    rows = (n_models + 1) // 2

    view = py3Dmol.view(
        viewergrid=(rows, cols),
        width=1200,
        height=500 * rows
    )

    for i, pdb_path in enumerate(pdb_files):
        r, c = i // cols, i % cols
        with open(pdb_path) as f:
            pdb_data = f.read()

        view.addModel(pdb_data, 'pdb', viewer=(r, c))
        # E2A = chain A (yellow), HPr = chain B (cyan)
        view.setStyle({'chain': 'A'}, {'cartoon': {'color': 'yellow'}}, viewer=(r, c))
        view.setStyle({'chain': 'B'}, {'cartoon': {'color': 'cyan'}}, viewer=(r, c))
        view.zoomTo(viewer=(r, c))

        score_label = pdb_path.stem
        print(f'[{r},{c}] {score_label}')

    view.show()

## 9. Overlay Best Model vs. Reference Complex

In [ ]:
import py3Dmol
from pathlib import Path

run_dir = Path(exercise_path) / 'run1-e2a-hpr'
ref_pdb = os.path.join(data_dir, 'e2a-hpr_1GGR.pdb')

emref_dirs = sorted(run_dir.glob('*_emref')) or sorted(run_dir.glob('*_flexref'))
top_model = sorted(emref_dirs[-1].glob('*.pdb'))[0] if emref_dirs else None

if top_model:
    view = py3Dmol.view(width=800, height=600)

    # Best docked model
    with open(top_model) as f:
        view.addModel(f.read(), 'pdb')
    view.setStyle({'model': 0, 'chain': 'A'}, {'cartoon': {'color': 'yellow'}})
    view.setStyle({'model': 0, 'chain': 'B'}, {'cartoon': {'color': 'cyan'}})

    # Reference complex
    with open(ref_pdb) as f:
        view.addModel(f.read(), 'pdb')
    view.setStyle({'model': 1}, {'cartoon': {'color': 'orange', 'opacity': 0.5}})

    view.zoomTo()
    view.show()
    print('Yellow/Cyan = Best docked model | Orange (transparent) = Reference 1GGR')
else:
    print('No model found — check run completed.')